<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/07_TADP_Experiment_F_Governance_Fault_Injection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ======================================================================================
# 07_TADP_Experiment_F_Governance_Fault_Injection.py
# TADP Experiment F — Governance Fault-Injection / Policy Boundary Validation
# FINAL policy-aligned governance-only experiment
# ======================================================================================
#
# PURPOSE
# -------
# Validate fail-closed enforcement and exact decision-path behavior of the FINAL TADP
# admission policy. This experiment is governance-only: it does not train a model and
# does not use predictive TEST outcomes.
#
# FINAL POLICY UNDER TEST
# -----------------------
# 1) Evidence-validity gate:
#      every active factor score must be present, numeric, finite, and strictly > 0.
# 2) Dimension floor:
#      every active dimension score must satisfy the configured minimum.
# 3) HPS decision bands:
#      HPS below the lower cut -> Reject
#      HPS at/above the upper cut -> Direct Accept
#      otherwise -> automated critical-dimension review
# 4) HPS_review:
#      critical-dimension review score at/above its threshold -> Accept
#      otherwise -> Reject
#
# DESIGN
# ------
# * K = 20 controlled contributors.
# * 20 independent injection seeds.
# * Fault prevalence = 10%, 30%, and 50% of contributors.
# * 30 policy test cases:
#     - evidence validity: missing, non-finite, zero, negative, epsilon-positive control
#     - dimension floor: below / exactly at / just above, separately for D1..D6
#     - HPS lower boundary: just below / exactly at
#     - HPS upper boundary: just below / exactly at
#     - HPS_review boundary: just below / exactly at
#     - clean strong control
# * 30 x 3 x 20 = 1,800 experimental conditions.
# * 1,800 x 20 = 36,000 client-level governance decisions.
#
# IMPORTANT
# ---------
# These are controlled score-level policy tests. Boundary profiles intentionally use
# small decimal offsets so that the implementation comparator itself is tested exactly.
# They are not estimates of real-world fault prevalence.
# ======================================================================================

import os
import json
import math
import hashlib
import zipfile
from pathlib import Path
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd


# ======================================================================================
# FINAL TADP POLICY CONSTANTS
# ======================================================================================

EXPERIMENT_VERSION = (
    "TADP-F-FINAL-28F-HPSREVIEW-GOVERNANCE-FAULT-INJECTION-"
    "K20-20SEED-3PREV-1800COND"
)

GOOD_CUT = 3.0
HIGH_CUT = 3.5
DIMENSION_MIN_FLOOR = 2.5
HPS_REVIEW_THRESHOLD = 3.25
EVIDENCE_VALIDITY_MIN_EXCLUSIVE = 0.0
MAX_FACTOR_SCORE = 5.0

WEIGHTS_HPS = {
    "dim1": 0.25,
    "dim2": 0.15,
    "dim3": 0.10,
    "dim4": 0.10,
    "dim5": 0.30,
    "dim6": 0.10,
}

CRITICAL_REVIEW_DIMS = ("dim1", "dim2", "dim5", "dim6")
CRITICAL_REVIEW_WEIGHTS = {
    d: WEIGHTS_HPS[d] / sum(WEIGHTS_HPS[q] for q in CRITICAL_REVIEW_DIMS)
    for d in CRITICAL_REVIEW_DIMS
}

EXPECTED_CRITICAL_REVIEW_WEIGHTS = {
    "dim1": 0.3125,
    "dim2": 0.1875,
    "dim5": 0.3750,
    "dim6": 0.1250,
}

# Use the exact mathematically renormalized weights for boundary testing so that
# equality at HPS_review is not changed by floating-point normalization noise.
CRITICAL_REVIEW_WEIGHTS = dict(EXPECTED_CRITICAL_REVIEW_WEIGHTS)

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

assert abs(sum(WEIGHTS_HPS.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert all(
    math.isclose(
        CRITICAL_REVIEW_WEIGHTS[d],
        EXPECTED_CRITICAL_REVIEW_WEIGHTS[d],
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    for d in CRITICAL_REVIEW_DIMS
)


# ======================================================================================
# EXPERIMENT-F DESIGN
# ======================================================================================

K_SUBMISSIONS = 20
CLIENT_IDS = [f"C{i:02d}" for i in range(1, K_SUBMISSIONS + 1)]
INJECTION_SEEDS = list(range(2001, 2021))  # 20 independent injection seeds
FAULT_PREVALENCE = [0.10, 0.30, 0.50]

STRONG_SCORE = 4.5
BOUNDARY_EPS = 1e-3
POSITIVE_EPS = 1e-6

OUTPUT_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
EXPERIMENT_ROOT = OUTPUT_ROOT / f"TADP_EXPERIMENT_F_{EXPERIMENT_VERSION}"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

ALL_FACTOR_KEYS = [
    (d, f)
    for d in FACTOR_NAMES
    for f in FACTOR_NAMES[d]
]


# ======================================================================================
# FINAL POLICY IMPLEMENTATION
# ======================================================================================

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    out = {}
    for dim, names in FACTOR_NAMES.items():
        vals = []
        for factor in names:
            if factor not in factors.get(dim, {}):
                out[dim] = np.nan
                break
            try:
                value = float(factors[dim][factor])
            except Exception:
                out[dim] = np.nan
                break
            if not np.isfinite(value):
                out[dim] = np.nan
                break
            vals.append(value)
        else:
            out[dim] = float(np.mean(vals))
    return out


def evidence_validity_audit(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, Any]:
    missing, nonfinite, nonpositive = [], [], []

    for dim, names in FACTOR_NAMES.items():
        for factor in names:
            key = f"{dim}.{factor}"
            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            try:
                value = float(factors[dim][factor])
            except Exception:
                nonfinite.append(key)
                continue

            if not np.isfinite(value):
                nonfinite.append(key)
            elif value <= EVIDENCE_VALIDITY_MIN_EXCLUSIVE:
                nonpositive.append(key)

    return {
        "passed": not (missing or nonfinite or nonpositive),
        "missing": missing,
        "nonfinite": nonfinite,
        "nonpositive": nonpositive,
    }


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_HPS,
) -> float:
    if not all(np.isfinite(float(dimensions[d])) for d in weights):
        return np.nan
    return float(sum(weights[d] * float(dimensions[d]) for d in weights))


def compute_hps_review(
    dimensions: Dict[str, float]
) -> float:
    if not all(
        np.isfinite(float(dimensions[d]))
        for d in CRITICAL_REVIEW_DIMS
    ):
        return np.nan

    return float(
        sum(
            CRITICAL_REVIEW_WEIGHTS[d] * float(dimensions[d])
            for d in CRITICAL_REVIEW_DIMS
        )
    )


def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    return [
        dim
        for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    validity = evidence_validity_audit(factors)
    dimensions = dimension_scores_from_factors(factors)
    hps = compute_hps(dimensions)
    hps_review = compute_hps_review(dimensions)
    floor_failures = (
        dimension_floor_failures(dimensions)
        if all(np.isfinite(float(v)) for v in dimensions.values())
        else []
    )

    base = {
        "evidence_validity_pass": bool(validity["passed"]),
        "evidence_missing": ";".join(validity["missing"]),
        "evidence_nonfinite": ";".join(validity["nonfinite"]),
        "evidence_nonpositive": ";".join(validity["nonpositive"]),
        "hps": float(hps) if np.isfinite(hps) else np.nan,
        "hps_review": (
            float(hps_review) if np.isfinite(hps_review) else np.nan
        ),
        "dimension_floor_failures": ";".join(floor_failures),
        "dimensions": dimensions,
    }

    if not validity["passed"]:
        return {
            **base,
            "decision_path": "EVIDENCE_VALIDITY_GATE",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_EVIDENCE_VALIDITY",
        }

    if floor_failures:
        return {
            **base,
            "decision_path": "DIMENSION_FLOOR",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_DIMENSION_FLOOR",
        }

    if hps < GOOD_CUT:
        return {
            **base,
            "decision_path": "LOW_HPS",
            "final_action": "REJECT",
            "status": "AUTO_REJECTED_LOW_HPS",
        }

    if hps >= HIGH_CUT:
        return {
            **base,
            "decision_path": "DIRECT_AUTO_ACCEPT",
            "final_action": "ACCEPT",
            "status": "DIRECT_AUTO_ACCEPTED",
        }

    if hps_review >= HPS_REVIEW_THRESHOLD:
        return {
            **base,
            "decision_path": "HPS_REVIEW_BAND",
            "final_action": "ACCEPT",
            "status": "ACCEPTED_AFTER_AUTOMATED_REVIEW",
        }

    return {
        **base,
        "decision_path": "HPS_REVIEW_BAND",
        "final_action": "REJECT",
        "status": "AUTO_REJECTED_HPS_REVIEW",
    }


# ======================================================================================
# CONTROLLED PROFILES
# ======================================================================================

def uniform_profile(
    dimension_values: Dict[str, float]
) -> Dict[str, Dict[str, float]]:
    return {
        dim: {
            factor: float(dimension_values[dim])
            for factor in FACTOR_NAMES[dim]
        }
        for dim in FACTOR_NAMES
    }


def strong_profile() -> Dict[str, Dict[str, float]]:
    return uniform_profile(
        {dim: STRONG_SCORE for dim in FACTOR_NAMES}
    )


def copy_profile(
    profile: Dict[str, Dict[str, float]]
) -> Dict[str, Dict[str, float]]:
    return {
        dim: dict(values)
        for dim, values in profile.items()
    }


def profile_for_case(
    case: Dict[str, Any],
    target_factor: Tuple[str, str],
) -> Dict[str, Dict[str, float]]:
    case_id = case["case_id"]
    profile = strong_profile()

    if case_id == "EVIDENCE_MISSING":
        d, f = target_factor
        del profile[d][f]
        return profile

    if case_id == "EVIDENCE_NONFINITE":
        d, f = target_factor
        profile[d][f] = np.nan
        return profile

    if case_id == "EVIDENCE_ZERO":
        d, f = target_factor
        profile[d][f] = 0.0
        return profile

    if case_id == "EVIDENCE_NEGATIVE":
        d, f = target_factor
        profile[d][f] = -BOUNDARY_EPS
        return profile

    if case_id == "EVIDENCE_EPSILON_POSITIVE":
        d, f = target_factor
        profile[d][f] = POSITIVE_EPS
        return profile

    if case["category"] == "DIMENSION_FLOOR":
        target_dim = case["target_dim"]
        if case["boundary_relation"] == "BELOW":
            value = DIMENSION_MIN_FLOOR - BOUNDARY_EPS
        elif case["boundary_relation"] == "AT":
            value = DIMENSION_MIN_FLOOR
        elif case["boundary_relation"] == "ABOVE":
            value = DIMENSION_MIN_FLOOR + BOUNDARY_EPS
        else:
            raise ValueError(case)
        for f in FACTOR_NAMES[target_dim]:
            profile[target_dim][f] = float(value)
        return profile

    if case_id == "HPS_LOWER_BELOW":
        return uniform_profile({
            dim: GOOD_CUT - BOUNDARY_EPS
            for dim in FACTOR_NAMES
        })

    if case_id == "HPS_LOWER_AT":
        return uniform_profile({
            dim: GOOD_CUT
            for dim in FACTOR_NAMES
        })

    if case_id == "HPS_UPPER_BELOW":
        return uniform_profile({
            dim: HIGH_CUT - BOUNDARY_EPS
            for dim in FACTOR_NAMES
        })

    if case_id == "HPS_UPPER_AT":
        return uniform_profile({
            dim: HIGH_CUT
            for dim in FACTOR_NAMES
        })

    if case_id in {"HPS_REVIEW_BELOW", "HPS_REVIEW_AT"}:
        critical_value = (
            HPS_REVIEW_THRESHOLD - BOUNDARY_EPS
            if case_id == "HPS_REVIEW_BELOW"
            else HPS_REVIEW_THRESHOLD
        )
        dims = {
            dim: (
                critical_value
                if dim in CRITICAL_REVIEW_DIMS
                else 4.0
            )
            for dim in FACTOR_NAMES
        }
        return uniform_profile(dims)

    if case_id == "CLEAN_STRONG":
        return strong_profile()

    raise ValueError(f"Unknown test case: {case_id}")


def build_test_cases() -> List[Dict[str, Any]]:
    cases = [
        {
            "case_id": "EVIDENCE_MISSING",
            "category": "EVIDENCE_VALIDITY",
            "boundary_relation": "FAULT",
            "expected_action": "REJECT",
            "expected_path": "EVIDENCE_VALIDITY_GATE",
        },
        {
            "case_id": "EVIDENCE_NONFINITE",
            "category": "EVIDENCE_VALIDITY",
            "boundary_relation": "FAULT",
            "expected_action": "REJECT",
            "expected_path": "EVIDENCE_VALIDITY_GATE",
        },
        {
            "case_id": "EVIDENCE_ZERO",
            "category": "EVIDENCE_VALIDITY",
            "boundary_relation": "FAULT",
            "expected_action": "REJECT",
            "expected_path": "EVIDENCE_VALIDITY_GATE",
        },
        {
            "case_id": "EVIDENCE_NEGATIVE",
            "category": "EVIDENCE_VALIDITY",
            "boundary_relation": "FAULT",
            "expected_action": "REJECT",
            "expected_path": "EVIDENCE_VALIDITY_GATE",
        },
        {
            "case_id": "EVIDENCE_EPSILON_POSITIVE",
            "category": "EVIDENCE_VALIDITY",
            "boundary_relation": "ABOVE",
            "expected_action": "ACCEPT",
            "expected_path": "DIRECT_AUTO_ACCEPT",
        },
    ]

    for dim in FACTOR_NAMES:
        cases.extend([
            {
                "case_id": f"DIMENSION_FLOOR_{dim.upper()}_BELOW",
                "category": "DIMENSION_FLOOR",
                "target_dim": dim,
                "boundary_relation": "BELOW",
                "expected_action": "REJECT",
                "expected_path": "DIMENSION_FLOOR",
            },
            {
                "case_id": f"DIMENSION_FLOOR_{dim.upper()}_AT",
                "category": "DIMENSION_FLOOR",
                "target_dim": dim,
                "boundary_relation": "AT",
                "expected_action": "ACCEPT",
                "expected_path": "DIRECT_AUTO_ACCEPT",
            },
            {
                "case_id": f"DIMENSION_FLOOR_{dim.upper()}_ABOVE",
                "category": "DIMENSION_FLOOR",
                "target_dim": dim,
                "boundary_relation": "ABOVE",
                "expected_action": "ACCEPT",
                "expected_path": "DIRECT_AUTO_ACCEPT",
            },
        ])

    cases.extend([
        {
            "case_id": "HPS_LOWER_BELOW",
            "category": "HPS_LOWER",
            "boundary_relation": "BELOW",
            "expected_action": "REJECT",
            "expected_path": "LOW_HPS",
        },
        {
            "case_id": "HPS_LOWER_AT",
            "category": "HPS_LOWER",
            "boundary_relation": "AT",
            "expected_action": "REJECT",
            "expected_path": "HPS_REVIEW_BAND",
        },
        {
            "case_id": "HPS_UPPER_BELOW",
            "category": "HPS_UPPER",
            "boundary_relation": "BELOW",
            "expected_action": "ACCEPT",
            "expected_path": "HPS_REVIEW_BAND",
        },
        {
            "case_id": "HPS_UPPER_AT",
            "category": "HPS_UPPER",
            "boundary_relation": "AT",
            "expected_action": "ACCEPT",
            "expected_path": "DIRECT_AUTO_ACCEPT",
        },
        {
            "case_id": "HPS_REVIEW_BELOW",
            "category": "HPS_REVIEW",
            "boundary_relation": "BELOW",
            "expected_action": "REJECT",
            "expected_path": "HPS_REVIEW_BAND",
        },
        {
            "case_id": "HPS_REVIEW_AT",
            "category": "HPS_REVIEW",
            "boundary_relation": "AT",
            "expected_action": "ACCEPT",
            "expected_path": "HPS_REVIEW_BAND",
        },
        {
            "case_id": "CLEAN_STRONG",
            "category": "CONTROL",
            "boundary_relation": "CONTROL",
            "expected_action": "ACCEPT",
            "expected_path": "DIRECT_AUTO_ACCEPT",
        },
    ])

    if len(cases) != 30:
        raise RuntimeError(
            f"Expected 30 Experiment-F test cases, got {len(cases)}"
        )
    return cases


TEST_CASES = build_test_cases()


# ======================================================================================
# EXPERIMENT EXECUTION
# ======================================================================================

def stable_rng_seed(*parts: Any) -> int:
    text = "|".join(map(str, parts))
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def choose_affected_clients(
    seed: int,
    prevalence: float,
    case_id: str,
) -> List[str]:
    n = int(round(K_SUBMISSIONS * float(prevalence)))
    n = max(1, min(K_SUBMISSIONS, n))
    rng = np.random.default_rng(
        stable_rng_seed(EXPERIMENT_VERSION, seed, prevalence, case_id)
    )
    return sorted(
        map(str, rng.choice(CLIENT_IDS, size=n, replace=False))
    )


def choose_target_factor(
    seed: int,
    case_id: str,
) -> Tuple[str, str]:
    rng = np.random.default_rng(
        stable_rng_seed("factor", seed, case_id)
    )
    idx = int(rng.integers(0, len(ALL_FACTOR_KEYS)))
    return ALL_FACTOR_KEYS[idx]


def run_condition(
    injection_seed: int,
    prevalence: float,
    case: Dict[str, Any],
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    affected = choose_affected_clients(
        injection_seed, prevalence, case["case_id"]
    )
    affected_set = set(affected)
    target_factor = choose_target_factor(
        injection_seed, case["case_id"]
    )

    rows = []
    for cid in CLIENT_IDS:
        is_affected = cid in affected_set
        if is_affected:
            factors = profile_for_case(case, target_factor)
        else:
            factors = strong_profile()

        decision = tadp_decision(factors)
        row = {
            "experiment_version": EXPERIMENT_VERSION,
            "injection_seed": int(injection_seed),
            "prevalence": float(prevalence),
            "case_id": case["case_id"],
            "category": case["category"],
            "boundary_relation": case["boundary_relation"],
            "target_dimension": case.get("target_dim", ""),
            "target_factor": (
                f"{target_factor[0]}.{target_factor[1]}"
                if case["category"] == "EVIDENCE_VALIDITY"
                else ""
            ),
            "client": cid,
            "affected": bool(is_affected),
            "expected_action_if_affected": case["expected_action"],
            "expected_path_if_affected": case["expected_path"],
            "evidence_validity_pass": decision["evidence_validity_pass"],
            "evidence_missing": decision["evidence_missing"],
            "evidence_nonfinite": decision["evidence_nonfinite"],
            "evidence_nonpositive": decision["evidence_nonpositive"],
            "hps": decision["hps"],
            "hps_review": decision["hps_review"],
            "dimension_floor_failures": decision[
                "dimension_floor_failures"
            ],
            "decision_path": decision["decision_path"],
            "final_action": decision["final_action"],
            "status": decision["status"],
        }
        for dim, value in decision["dimensions"].items():
            row[f"{dim}_score_0_5"] = value

        row["matches_expected_if_affected"] = (
            (not is_affected)
            or (
                row["final_action"] == case["expected_action"]
                and row["decision_path"] == case["expected_path"]
            )
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    affected_df = df[df["affected"]]
    honest_df = df[~df["affected"]]

    summary = {
        "experiment_version": EXPERIMENT_VERSION,
        "injection_seed": int(injection_seed),
        "prevalence": float(prevalence),
        "case_id": case["case_id"],
        "category": case["category"],
        "boundary_relation": case["boundary_relation"],
        "target_dimension": case.get("target_dim", ""),
        "target_factor": (
            f"{target_factor[0]}.{target_factor[1]}"
            if case["category"] == "EVIDENCE_VALIDITY"
            else ""
        ),
        "affected_count": int(len(affected_df)),
        "honest_count": int(len(honest_df)),
        "expected_action": case["expected_action"],
        "expected_path": case["expected_path"],
        "affected_correct_count": int(
            affected_df["matches_expected_if_affected"].sum()
        ),
        "affected_behavior_accuracy": float(
            affected_df["matches_expected_if_affected"].mean()
        ),
        "affected_rejected_count": int(
            affected_df["final_action"].eq("REJECT").sum()
        ),
        "affected_accepted_count": int(
            affected_df["final_action"].eq("ACCEPT").sum()
        ),
        "honest_false_rejection_count": int(
            honest_df["final_action"].eq("REJECT").sum()
        ),
        "honest_false_rejection_rate": (
            float(honest_df["final_action"].eq("REJECT").mean())
            if len(honest_df)
            else 0.0
        ),
        "condition_pass": bool(
            affected_df["matches_expected_if_affected"].all()
            and honest_df["final_action"].eq("ACCEPT").all()
        ),
    }
    return df, summary


def aggregate_results(
    decisions: pd.DataFrame,
    conditions: pd.DataFrame,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    by_case = (
        conditions.groupby(
            [
                "case_id",
                "category",
                "boundary_relation",
                "expected_action",
                "expected_path",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            conditions=("condition_pass", "size"),
            passed_conditions=("condition_pass", "sum"),
            affected_clients=("affected_count", "sum"),
            affected_behavior_accuracy=("affected_behavior_accuracy", "mean"),
            honest_false_rejection_rate=("honest_false_rejection_rate", "mean"),
        )
    )
    by_case["condition_pass_rate"] = (
        by_case["passed_conditions"] / by_case["conditions"]
    )

    affected = decisions[decisions["affected"]].copy()
    honest = decisions[~decisions["affected"]].copy()

    expected_reject = affected[
        affected["expected_action_if_affected"].eq("REJECT")
    ]
    expected_accept = affected[
        affected["expected_action_if_affected"].eq("ACCEPT")
    ]

    metrics = {
        "experiment_version": EXPERIMENT_VERSION,
        "K": K_SUBMISSIONS,
        "injection_seed_count": len(INJECTION_SEEDS),
        "prevalence_levels": FAULT_PREVALENCE,
        "test_case_count": len(TEST_CASES),
        "experimental_conditions": int(len(conditions)),
        "client_level_decisions": int(len(decisions)),
        "all_conditions_pass": bool(conditions["condition_pass"].all()),
        "condition_pass_rate": float(conditions["condition_pass"].mean()),
        "expected_reject_cases_affected_clients": int(len(expected_reject)),
        "expected_reject_recall": float(
            expected_reject["final_action"].eq("REJECT").mean()
        ),
        "expected_accept_cases_affected_clients": int(len(expected_accept)),
        "expected_accept_rate": float(
            expected_accept["final_action"].eq("ACCEPT").mean()
        ),
        "honest_client_decisions": int(len(honest)),
        "honest_false_rejection_rate": float(
            honest["final_action"].eq("REJECT").mean()
        ),
        "decision_path_accuracy_on_affected": float(
            (
                affected["decision_path"]
                == affected["expected_path_if_affected"]
            ).mean()
        ),
    }
    return by_case, metrics


# ======================================================================================
# AUDITS / PACKAGING
# ======================================================================================

def policy_self_test() -> Dict[str, Any]:
    checks = {}

    # Strong profile -> direct accept.
    d = tadp_decision(strong_profile())
    checks["strong_direct_accept"] = (
        d["decision_path"] == "DIRECT_AUTO_ACCEPT"
        and d["final_action"] == "ACCEPT"
    )

    # Review boundary just below -> reject.
    below_case = next(
        x for x in TEST_CASES if x["case_id"] == "HPS_REVIEW_BELOW"
    )
    d = tadp_decision(
        profile_for_case(below_case, ALL_FACTOR_KEYS[0])
    )
    checks["review_below_reject"] = (
        d["decision_path"] == "HPS_REVIEW_BAND"
        and d["final_action"] == "REJECT"
    )

    # Review boundary exactly at -> accept.
    at_case = next(
        x for x in TEST_CASES if x["case_id"] == "HPS_REVIEW_AT"
    )
    d = tadp_decision(
        profile_for_case(at_case, ALL_FACTOR_KEYS[0])
    )
    checks["review_at_accept"] = (
        d["decision_path"] == "HPS_REVIEW_BAND"
        and d["final_action"] == "ACCEPT"
    )

    # Upper HPS exactly at -> direct accept.
    at_upper = next(
        x for x in TEST_CASES if x["case_id"] == "HPS_UPPER_AT"
    )
    d = tadp_decision(
        profile_for_case(at_upper, ALL_FACTOR_KEYS[0])
    )
    checks["upper_at_direct_accept"] = (
        d["decision_path"] == "DIRECT_AUTO_ACCEPT"
        and d["final_action"] == "ACCEPT"
    )

    checks["all_pass"] = bool(all(checks.values()))
    return checks


def package_results(output_dir: Path) -> Path:
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })

    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv",
        index=False,
    )

    zip_path = (
        output_dir.parent
        / f"{output_dir.name}_RESULTS.zip"
    )
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(
                    p,
                    arcname=str(p.relative_to(output_dir)),
                )
    return zip_path


def main() -> Path:
    print("=" * 108)
    print(EXPERIMENT_VERSION)
    print("=" * 108)
    print(
        "Final policy: evidence validity -> dimension floor -> "
        "HPS bands -> HPS_review -> ACCEPT/REJECT"
    )
    print(
        f"K={K_SUBMISSIONS} | injection seeds={len(INJECTION_SEEDS)} | "
        f"prevalence={FAULT_PREVALENCE} | cases={len(TEST_CASES)}"
    )

    design = {
        "experiment_version": EXPERIMENT_VERSION,
        "purpose": (
            "governance-only fault injection and exact policy-boundary validation"
        ),
        "K": K_SUBMISSIONS,
        "injection_seeds": INJECTION_SEEDS,
        "prevalence_levels": FAULT_PREVALENCE,
        "test_case_count": len(TEST_CASES),
        "expected_conditions": (
            len(INJECTION_SEEDS)
            * len(FAULT_PREVALENCE)
            * len(TEST_CASES)
        ),
        "expected_client_level_decisions": (
            len(INJECTION_SEEDS)
            * len(FAULT_PREVALENCE)
            * len(TEST_CASES)
            * K_SUBMISSIONS
        ),
        "policy": {
            "evidence_validity": "all 28 factor scores present, finite, and >0",
            "dimension_floor": DIMENSION_MIN_FLOOR,
            "hps_lower": GOOD_CUT,
            "hps_upper": HIGH_CUT,
            "hps_review_threshold": HPS_REVIEW_THRESHOLD,
            "critical_review_dimensions": CRITICAL_REVIEW_DIMS,
            "critical_review_weights": CRITICAL_REVIEW_WEIGHTS,
        },
        "boundary_epsilon": BOUNDARY_EPS,
        "positive_epsilon_control": POSITIVE_EPS,
        "training_performed": False,
        "test_data_used": False,
        "interpretation": (
            "controlled score-level validation of the governance engine; "
            "not a real-world fault-prevalence estimate"
        ),
    }

    with open(
        EXPERIMENT_ROOT / "experiment_design.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(design, f, indent=2, sort_keys=True)

    pd.DataFrame(TEST_CASES).to_csv(
        EXPERIMENT_ROOT / "policy_test_cases.csv",
        index=False,
    )

    self_test = policy_self_test()
    with open(
        EXPERIMENT_ROOT / "policy_self_test.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(self_test, f, indent=2, sort_keys=True)

    if not self_test["all_pass"]:
        raise RuntimeError(
            f"Policy self-test failed: {self_test}"
        )

    decision_frames = []
    condition_rows = []

    total_conditions = (
        len(INJECTION_SEEDS)
        * len(FAULT_PREVALENCE)
        * len(TEST_CASES)
    )
    counter = 0

    for seed in INJECTION_SEEDS:
        for prevalence in FAULT_PREVALENCE:
            for case in TEST_CASES:
                counter += 1
                decisions, summary = run_condition(
                    seed, prevalence, case
                )
                decision_frames.append(decisions)
                condition_rows.append(summary)

                if counter % 100 == 0 or counter == total_conditions:
                    print(
                        f"Experiment F conditions: "
                        f"{counter}/{total_conditions} complete"
                    )

    decisions = pd.concat(
        decision_frames,
        ignore_index=True,
    )
    conditions = pd.DataFrame(condition_rows)

    by_case, metrics = aggregate_results(
        decisions,
        conditions,
    )

    if len(conditions) != 1800:
        raise RuntimeError(
            f"Expected 1,800 conditions, got {len(conditions)}"
        )
    if len(decisions) != 36000:
        raise RuntimeError(
            f"Expected 36,000 client-level decisions, got {len(decisions)}"
        )

    decisions.to_csv(
        EXPERIMENT_ROOT / "fault_injection_all_decisions.csv",
        index=False,
    )
    conditions.to_csv(
        EXPERIMENT_ROOT / "fault_injection_condition_summary.csv",
        index=False,
    )
    by_case.to_csv(
        EXPERIMENT_ROOT / "policy_boundary_summary_by_case.csv",
        index=False,
    )

    with open(
        EXPERIMENT_ROOT / "aggregate_metrics.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(metrics, f, indent=2, sort_keys=True)

    print("\n" + "=" * 108)
    print("EXPERIMENT F — FINAL POLICY AUDIT")
    print("=" * 108)
    print(json.dumps(metrics, indent=2, sort_keys=True))

    if not metrics["all_conditions_pass"]:
        bad = conditions.loc[
            ~conditions["condition_pass"]
        ].copy()
        bad.to_csv(
            EXPERIMENT_ROOT / "FAILED_CONDITIONS.csv",
            index=False,
        )
        raise RuntimeError(
            f"Fail-closed: {len(bad)} Experiment-F conditions "
            f"did not match expected policy behavior."
        )

    zip_path = package_results(EXPERIMENT_ROOT)

    print("\n" + "=" * 108)
    print(f"{EXPERIMENT_VERSION} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(
        f"ZIP size: "
        f"{zip_path.stat().st_size / (1024 ** 2):.2f} MiB"
    )
    print("=" * 108)

    # Automatic Colab download when available.
    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


if __name__ == "__main__":
    main()


TADP-F-FINAL-28F-HPSREVIEW-GOVERNANCE-FAULT-INJECTION-K20-20SEED-3PREV-1800COND
Final policy: evidence validity -> dimension floor -> HPS bands -> HPS_review -> ACCEPT/REJECT
K=20 | injection seeds=20 | prevalence=[0.1, 0.3, 0.5] | cases=30
Experiment F conditions: 100/1800 complete
Experiment F conditions: 200/1800 complete
Experiment F conditions: 300/1800 complete
Experiment F conditions: 400/1800 complete
Experiment F conditions: 500/1800 complete
Experiment F conditions: 600/1800 complete
Experiment F conditions: 700/1800 complete
Experiment F conditions: 800/1800 complete
Experiment F conditions: 900/1800 complete
Experiment F conditions: 1000/1800 complete
Experiment F conditions: 1100/1800 complete
Experiment F conditions: 1200/1800 complete
Experiment F conditions: 1300/1800 complete
Experiment F conditions: 1400/1800 complete
Experiment F conditions: 1500/1800 complete
Experiment F conditions: 1600/1800 complete
Experiment F conditions: 1700/1800 complete
Experiment F conditi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>